# 🏦 Banking Intent Classification — Fine-tuning with Unsloth

**Dataset:** BANKING77 (PolyAI)  
**Model:** Llama-3.2-1B-Instruct (4-bit QLoRA via Unsloth)  
**Platform:** Kaggle (GPU T4)  

> Notebook này thực hiện end-to-end: cài đặt → tiền xử lý → train → đánh giá → lưu checkpoint.

## 1. Cài đặt dependencies

In [1]:
# 1. Cài đặt các thư viện lõi tối ưu phần cứng
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton -q

# 2. Cài đặt Unsloth bản mới nhất từ Github
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q

# 3. Ghim ĐÚNG phiên bản "điểm ngọt" mà Unsloth yêu cầu (Giải quyết lỗi đỏ vừa rồi)
!pip install "transformers<=5.5.0" "datasets<4.4.0" "dill>=0.3.9" pyyaml scikit-learn -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Import thư viện

In [2]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print('✅ All imports OK')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ All imports OK


## 3. Cấu hình Hyperparameters

Tất cả hyperparameters được khai báo tập trung tại đây.

In [3]:
CONFIG = {
    # ── Model ──
    "model_name": "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "max_seq_length": 512,
    "load_in_4bit": True,

    # ── LoRA ──
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0,
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],

    # ── Training ──
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 4,  # effective batch = 32
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "optimizer": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "warmup_steps": 10,
    "max_grad_norm": 1.0,
    "logging_steps": 10,
    "seed": 42,

    # ── Save ──
    "output_dir": "./outputs",

    # ── Data sampling ──
    "train_per_class": 50,
    "test_per_class": 20,
}

print("📋 Configuration:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

📋 Configuration:
   model_name: unsloth/Llama-3.2-1B-Instruct-bnb-4bit
   max_seq_length: 512
   load_in_4bit: True
   lora_r: 16
   lora_alpha: 16
   lora_dropout: 0
   target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
   per_device_train_batch_size: 8
   gradient_accumulation_steps: 4
   num_train_epochs: 3
   learning_rate: 0.0002
   optimizer: adamw_8bit
   weight_decay: 0.01
   lr_scheduler_type: linear
   warmup_steps: 10
   max_grad_norm: 1.0
   logging_steps: 10
   seed: 42
   output_dir: ./outputs
   train_per_class: 50
   test_per_class: 20


## 4. Tải và tiền xử lý dữ liệu BANKING77

- Tải dataset từ HuggingFace bằng `load_dataset`
- Chuẩn hóa text (lowercase, strip)
- Ánh xạ label → intent name
- Lấy subset cân bằng theo class

In [4]:
# 77 intent labels theo thứ tự gốc của BANKING77
LABEL_NAMES = [
    "activate_my_card", "age_limit", "apple_pay_or_google_pay",
    "atm_support", "automatic_top_up",
    "balance_not_updated_after_bank_transfer",
    "balance_not_updated_after_cheque_or_cash_deposit",
    "beneficiary_not_allowed", "cancel_transfer", "card_about_to_expire",
    "card_acceptance", "card_arrival", "card_delivery_estimate",
    "card_linking", "card_not_working", "card_payment_fee_charged",
    "card_payment_not_recognised", "card_payment_wrong_exchange_rate",
    "card_swallowed", "cash_withdrawal_charge",
    "cash_withdrawal_not_recognised", "change_pin", "compromised_card",
    "contactless_not_working", "country_support", "declined_card_payment",
    "declined_cash_withdrawal", "declined_transfer",
    "direct_debit_payment_not_recognised", "disposable_card_limits",
    "edit_personal_details", "exchange_charge", "exchange_rate",
    "exchange_via_app", "extra_charge_on_statement", "failed_transfer",
    "fiat_currency_support", "freeze_card", "getting_spare_card",
    "getting_virtual_card", "get_disposable_virtual_card",
    "get_physical_card", "lost_or_stolen_card", "lost_or_stolen_phone",
    "order_physical_card", "passcode_forgotten", "pending_card_payment",
    "pending_cash_withdrawal", "pending_top_up", "pending_transfer",
    "pin_blocked", "receiving_money", "Refund_not_showing_up",
    "request_refund", "reverted_card_payment?",
    "supported_cards_and_currencies", "terminate_account",
    "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
    "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits",
    "top_up_reverted", "topping_up_by_card", "transaction_charged_twice",
    "transfer_fee_charged", "transfer_into_account",
    "transfer_not_received_by_recipient", "transfer_timing",
    "unable_to_verify_identity", "verify_my_identity",
    "verify_source_of_funds", "verify_top_up",
    "virtual_card_not_working", "visa_or_mastercard",
    "why_verify_identity", "wrong_amount_of_cash_received",
    "wrong_exchange_rate_for_cash_withdrawal",
]

print(f"Total intent classes: {len(LABEL_NAMES)}")

Total intent classes: 78


In [5]:
import pandas as pd
from datasets import load_dataset

# 1. Tải dataset bằng bản sao lưu Parquet để tránh lỗi bảo mật script trên Kaggle
print("📥 Loading BANKING77 via Parquet branch...")
raw = load_dataset("parquet", data_files={
    "train": "hf://datasets/PolyAI/banking77@refs/convert/parquet/default/train/*.parquet",
    "test": "hf://datasets/PolyAI/banking77@refs/convert/parquet/default/test/*.parquet"
})

print(f"   Train split: {len(raw['train'])} samples")
print(f"   Test split:  {len(raw['test'])} samples")

# 2. Xử lý dữ liệu
def build_df(split):
    """Chuyển split thành DataFrame, chuẩn hóa text."""
    return pd.DataFrame({
        "text": [r["text"].strip().lower() for r in split],
        "label": [r["label"] for r in split],
        "intent": [LABEL_NAMES[r["label"]] for r in split],
    })

train_full = build_df(raw["train"])
test_full  = build_df(raw["test"])

# 3. Lấy subset cân bằng theo class
train_df = (
    train_full.groupby("label", group_keys=False)
    .apply(lambda g: g.sample(
        n=min(CONFIG["train_per_class"], len(g)), random_state=42))
    .reset_index(drop=True)
)
test_df = (
    test_full.groupby("label", group_keys=False)
    .apply(lambda g: g.sample(
        n=min(CONFIG["test_per_class"], len(g)), random_state=42))
    .reset_index(drop=True)
)

# 4. In kết quả
print(f"\n✅ Subset created:")
print(f"   Train: {len(train_df)} samples ({len(train_df['label'].unique())} classes)")
print(f"   Test:  {len(test_df)} samples ({len(test_df['label'].unique())} classes)")
print(f"\n📊 Sample data:")
display(train_df.head()) # Dùng display() trên Kaggle để bảng hiện đẹp hơn print()

📥 Loading BANKING77 via Parquet branch...
   Train split: 10003 samples
   Test split:  3080 samples

✅ Subset created:
   Train: 3826 samples (77 classes)
   Test:  1540 samples (77 classes)

📊 Sample data:


,text,label,intent
0,"i need to activate my card, how is that done?",0,activate_my_card
1,"my new card has arrived, what's the activation...",0,activate_my_card
2,i need to do a card activation.,0,activate_my_card
3,"i am unable to activate my card, it won't let me.",0,activate_my_card
4,i tried activating my card and it didn't work,0,activate_my_card


## 5. Chuẩn bị dữ liệu cho SFT

Tạo cột instruction/input/output và lưu CSV.

In [6]:
INSTRUCTION = (
    "You are a banking intent classifier. "
    "Given a customer's message, classify it into one of the banking "
    "intent categories. Only output the intent label, nothing else."
)

# Thêm cột SFT
train_df["instruction"] = INSTRUCTION
train_df["input"]       = train_df["text"]
train_df["output"]      = train_df["intent"]
test_df["instruction"]  = INSTRUCTION
test_df["input"]        = test_df["text"]
test_df["output"]       = test_df["intent"]

# Lưu CSV
os.makedirs("sample_data", exist_ok=True)
train_df.to_csv("sample_data/train.csv", index=False)
test_df.to_csv("sample_data/test.csv", index=False)
print("✅ Saved sample_data/train.csv & test.csv")

✅ Saved sample_data/train.csv & test.csv


## 6. Tải model và gắn LoRA adapters

In [7]:
print("📦 Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=CONFIG["load_in_4bit"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=CONFIG["target_modules"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)
print("✅ Model loaded + LoRA attached")

📦 Loading model...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


✅ Model loaded + LoRA attached


## 7. Format dataset cho training (Alpaca prompt)

In [8]:
ALPACA_PROMPT = """Below is an instruction that describes a task, paired with \
an input that provides further context. Write a response that appropriately \
completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_func(examples):
    texts = []
    for instr, inp, out in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        texts.append(ALPACA_PROMPT.format(instr, inp, out) + EOS_TOKEN)
    return {"text": texts}

train_dataset = Dataset.from_pandas(train_df[["instruction", "input", "output"]])
train_dataset = train_dataset.map(formatting_func, batched=True)

print(f"📊 {len(train_dataset)} training samples formatted")
print(f"\n📝 Example prompt:\n{train_dataset[0]['text'][:500]}...")

Map:   0%|          | 0/3826 [00:00<?, ? examples/s]

📊 3826 training samples formatted

📝 Example prompt:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a banking intent classifier. Given a customer's message, classify it into one of the banking intent categories. Only output the intent label, nothing else.

### Input:
i need to activate my card, how is that done?

### Response:
activate_my_card<|eot_id|>...


## 8. Fine-tuning

Training với SFTTrainer, ghi log mỗi 10 steps.

In [10]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        num_train_epochs=CONFIG["num_train_epochs"],
        learning_rate=CONFIG["learning_rate"],
        optim=CONFIG["optimizer"],
        weight_decay=CONFIG["weight_decay"],
        lr_scheduler_type=CONFIG["lr_scheduler_type"],
        warmup_steps=CONFIG["warmup_steps"],
        max_grad_norm=CONFIG["max_grad_norm"],
        logging_steps=CONFIG["logging_steps"],
        output_dir=CONFIG["output_dir"],
        save_strategy="epoch",
        seed=CONFIG["seed"],
        fp16=True,
        report_to="none",
        average_tokens_across_devices=False,
    ),
)

print("🚀 Training started...\n")
stats = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Loss: {stats.metrics['train_loss']:.4f}")
print(f"   Time: {stats.metrics['train_runtime']:.0f}s")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/3826 [00:00<?, ? examples/s]

🚀 Training started...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,826 | Num Epochs = 3 | Total steps = 180
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,3.073877
20,0.963538
30,0.688400
40,0.567890
50,0.493963
60,0.430454
70,0.418052
80,0.408985
90,0.411688
100,0.363584


Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-180/tokenizer_config.json.



✅ Training complete!
   Loss: 0.5944
   Time: 394s


## 9. Lưu checkpoint

In [11]:
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"💾 Checkpoint saved to {CONFIG['output_dir']}/")

Unsloth: Restored added_tokens_decoder metadata in ./outputs/tokenizer_config.json.


💾 Checkpoint saved to ./outputs/


## 10. Đánh giá trên Test set

Chạy inference trên toàn bộ test set và tính accuracy + classification report.

In [13]:
print("📊 Evaluating on test set...\n")
FastLanguageModel.for_inference(model)

y_true, y_pred = [], []

for idx, row in test_df.iterrows():
    prompt = ALPACA_PROMPT.format(row["instruction"], row["input"], "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=32, max_length=None, temperature=0.0, do_sample=False,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predicted = response.split("### Response:")[-1].strip().split("\n")[0].strip()

    y_true.append(row["output"])
    y_pred.append(predicted)

    if (idx + 1) % 200 == 0:
        acc_so_far = accuracy_score(y_true, y_pred)
        print(f"   [{idx+1}/{len(test_df)}]  acc = {acc_so_far*100:.1f}%")

accuracy = accuracy_score(y_true, y_pred)
print(f"\n{'='*50}")
print(f"  🎯 TEST ACCURACY: {accuracy*100:.2f}%")
print(f"{'='*50}")
print(classification_report(y_true, y_pred, zero_division=0))

📊 Evaluating on test set...

   [200/1540]  acc = 83.5%
   [400/1540]  acc = 79.8%
   [600/1540]  acc = 79.3%
   [800/1540]  acc = 81.1%
   [1000/1540]  acc = 81.2%
   [1200/1540]  acc = 81.2%
   [1400/1540]  acc = 79.8%

  🎯 TEST ACCURACY: 79.94%
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.93      0.70      0.80        20
                                activate_my_card       0.95      0.90      0.92        20
                                       age_limit       0.91      1.00      0.95        20
                         apple_pay_or_google_pay       0.95      1.00      0.98        20
                                     atm_support       0.95      1.00      0.98        20
                                automatic_top_up       0.93      0.70      0.80        20
         balance_not_updated_after_bank_transfer       0.62      0.80      0.70        20
balance_not_updated_after_chequ

## 11. Quick inference test

In [ ]:
print("🧪 Quick inference test:\n")
test_msgs = [
    "I am still waiting on my card?",
    "How do I change my PIN number?",
    "I want to cancel my transfer",
    "Why was I charged extra for my card payment?",
    "I lost my card, what should I do?",
    "Is my card compatible with Apple Pay?",
]

for msg in test_msgs:
    prompt = ALPACA_PROMPT.format(INSTRUCTION, msg, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=32,
                         temperature=0.0, do_sample=False)
    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    pred = pred.split("### Response:")[-1].strip().split("\n")[0].strip()
    print(f'  "{msg}"  →  {pred}')

## 12. Lưu config files (train.yaml + inference.yaml)

In [ ]:
import yaml

# ── train.yaml ──
train_cfg = {
    "model_name": CONFIG["model_name"],
    "max_seq_length": CONFIG["max_seq_length"],
    "load_in_4bit": CONFIG["load_in_4bit"],
    "lora_r": CONFIG["lora_r"],
    "lora_alpha": CONFIG["lora_alpha"],
    "lora_dropout": CONFIG["lora_dropout"],
    "target_modules": CONFIG["target_modules"],
    "per_device_train_batch_size": CONFIG["per_device_train_batch_size"],
    "gradient_accumulation_steps": CONFIG["gradient_accumulation_steps"],
    "num_train_epochs": CONFIG["num_train_epochs"],
    "learning_rate": CONFIG["learning_rate"],
    "optimizer": CONFIG["optimizer"],
    "weight_decay": CONFIG["weight_decay"],
    "lr_scheduler_type": CONFIG["lr_scheduler_type"],
    "warmup_steps": CONFIG["warmup_steps"],
    "max_grad_norm": CONFIG["max_grad_norm"],
    "logging_steps": CONFIG["logging_steps"],
    "seed": CONFIG["seed"],
    "output_dir": CONFIG["output_dir"],
    "save_strategy": "epoch",
    "train_data_path": "./sample_data/train.csv",
    "test_data_path": "./sample_data/test.csv",
}

infer_cfg = {
    "model_path": "./outputs",
    "base_model": CONFIG["model_name"],
    "max_seq_length": CONFIG["max_seq_length"],
    "load_in_4bit": True,
    "max_new_tokens": 32,
    "temperature": 0.0,
}

os.makedirs("configs", exist_ok=True)
with open("configs/train.yaml", "w") as f:
    yaml.dump(train_cfg, f, default_flow_style=False, sort_keys=False)
with open("configs/inference.yaml", "w") as f:
    yaml.dump(infer_cfg, f, default_flow_style=False, sort_keys=False)

print("✅ Saved configs/train.yaml & configs/inference.yaml")

## 13. Nén output để tải về local

Chạy cell này để tạo file zip chứa checkpoint + data + configs.

In [ ]:
!zip -r /kaggle/working/banking_intent_outputs.zip outputs/ sample_data/ configs/
print("\n✅ File zip sẵn sàng tải về: banking_intent_outputs.zip")
print("   → Vào Output tab của notebook để download.")

## ✅ Hoàn thành!

**Bước tiếp theo:**
1. Tải `banking_intent_outputs.zip` từ Output tab
2. Giải nén vào thư mục project local
3. Chạy inference: `python scripts/inference.py configs/inference.yaml`
4. Quay video demo và cập nhật README